# 🎧 A7N_ref_FT_01 청음 — VoxCPM2 한국어 감정 FT

- 입력 규칙(D 재측정과 같음): **`(v1 지시문)문장` + 처음 보는 목표 화자의 무감정 reference** · prompt 없음 · 준비 합성 뒤 · seed 고정
- 무엇을 들을까: ① **목소리가 reference 와 같은 사람인가**(끝까지) ② **칸의 감정이 원본보다 분명한가**(A = 약 · B = 강) ③ 끊김 · 늘어짐 · 끝 잘림 · 발음
- 숫자(D 재측정 · TB)는 **참고용** — 귀가 정본. 비교 페이지: http://127.0.0.1:8765/vox_koft/d_eval/index.html
- 실행 순서: 1 → 2 → 3(ref) → 4(상위 5 step × 7칸) → `PICK_STEP` 고른 뒤 5 · 6 · 7

## 1. 환경

In [ ]:
import os, sys
# ⚙️ GPU — 청음은 몇 GB 만 씀. 학습 중인 GPU 와 같이 써도 되지만 비어 있는 GPU 를 먼저.
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '2')
sys.path.insert(0, os.path.expanduser('~/workspaces/vox_koft'))
from voxko import listen
RUN_NAME = 'A7N_ref_FT_01'
print('🎓', RUN_NAME, listen.run_info(RUN_NAME))
print('💾 저장된 step:', listen.steps(RUN_NAME))
print('LISTEN_SMOKE =', listen.SMOKE)

## 2. 노브 · ⭐ 들을 step

In [ ]:
SEED, CFG, TIMESTEPS = 1986, 2.0, 10        # D 재측정과 같음
# ⭐ best 5 (참고용 — 귀로 확정)
#   · D 재측정 참고 점수 상위(감정 + 목표1위(공정) − CER − 이상): s3000 140(감정 60% · 목표1위 W84% · 이상 2%) · s2250 138(감정 61% · 목표1위 W79% · 이상 4%) · s1500 136(감정 55% · 목표1위 W83% · 이상 4%) · s1000 134(감정 54% · 목표1위 W84% · 이상 3%) · s750 120(감정 51% · 목표1위 W79% · 이상 9%)
#   · 원본 기준선: 감정 47% · 목표1위 W67% · 이상 14% · 점수 99
#   · TB 처음 보는 화자 val loss 상위 5: [1000, 750, 2000, 500, 3000]
STEP_PICK_LIST = [3000, 2250, 1500, 1000, 750]
PICK_STEP = STEP_PICK_LIST[0]              # ← 4절을 들은 뒤 귀로 고른 step 으로 바꾸고 이 셀부터 다시 실행
L = listen.Listener(RUN_NAME, seed=SEED, cfg_value=CFG, timesteps=TIMESTEPS)

## 3. Reference — 처음 보는 화자 16명(무감정) 중 고르기
번호(`REF_IDX`)를 바꾸고 다시 실행하면 4절 이후가 그 목소리를 씁니다.

In [ ]:
listen.show_refs()
REF_IDX = 8                                 # 예: 8 = aihub_0012(남 · unseen8)
REF = listen.pick_ref(REF_IDX)
print('ref =', REF['speaker'], REF['gender'], REF['src'])

## 4. ⭐ 상위 5 step × 7칸 — 원본과 나란히
칸마다 같은 문장 · 같은 ref · 같은 seed. 첫 실행은 step 을 바꿔 가며 합성합니다(step 하나 로드 약 10초).

In [ ]:
L.grid(steps=[None] + STEP_PICK_LIST, emotions=listen.EMO7, ref=REF)

## 5. 고른 step(`PICK_STEP`) — 여러 목소리로
여자 · 남자 처음 보는 화자 몇 명으로 같은 7칸을 들어 봅니다(목소리가 reference 를 따라가는지).

In [ ]:
for i in [0, 4, 9, 12]:                     # 542 여 · 542 남 · unseen8 · unseen8
    L.grid(steps=[None, PICK_STEP], emotions=listen.EMO7, ref=listen.pick_ref(i))

## 6. 자유 문장 · 자유 지시문

In [ ]:
TEXT = '정말 고마워요. 덕분에 오늘 하루가 완전히 달라졌어요!'
EMOTION = 'Happy-B'                          # 표의 감정 키(v1 문구) — 또는 INSTRUCT 에 직접
INSTRUCT = None                              # 예: 'slightly faster, cheerful tone'
from IPython.display import Audio, display
for st in [None, PICK_STEP]:
    w, sr, full = L.synth(st, TEXT, emotion=None if INSTRUCT else EMOTION, instruct=INSTRUCT, ref=REF)
    print('원본' if st is None else f'step {st}', '·', full); display(Audio(w, rate=sr))

## 7. seed 3개 반복 — 같은 입력에서 얼마나 흔들리나

In [ ]:
for sd in ([1986] if listen.SMOKE else [1986, 1987, 1988]):
    L.grid(steps=[None, PICK_STEP], emotions=['Happy-B', 'Sad-B', 'Angry-B'], ref=REF, seed=sd)